# K-Means

Agrupamento nao-supervisionado que minimiza a soma dos quadrados
intra-cluster (inercia).

Usamos **k-means++** para inicializar centroides distantes entre si
(o que reduz bastante a chance de convergir para otimos locais ruins)
e repetimos o algoritmo `n_init` vezes, guardando o melhor resultado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

In [ ]:
class KMeans:
    """
    K-Means classico com inicializacao k-means++ e multiplos restarts.

    Algoritmo:
      1. Inicializa centroides (k-means++).
      2. Atribui cada ponto ao centroide mais proximo.
      3. Recalcula centroides como a media de cada cluster.
      4. Repete ate convergir (sem mudancas) ou atingir n_iters.
      5. Repete n_init vezes e fica com o melhor resultado (menor inercia).
    """

    def __init__(self, n_clusters=3, n_iters=300, n_init=10, tol=1e-4, random_state=None):
        if n_clusters <= 0:
            raise ValueError("n_clusters must be positive.")
        if n_init <= 0:
            raise ValueError("n_init must be positive.")

        self.n_clusters = n_clusters
        self.n_iters = n_iters
        self.n_init = n_init
        self.tol = tol
        self.random_state = random_state

        self.centroids_ = None
        self.labels_ = None
        self.inertia_ = None

    def _init_centroids(self, X, rng):
        """k-means++: escolhe centroides distantes entre si."""
        n_samples = X.shape[0]
        centroids = np.empty((self.n_clusters, X.shape[1]))
        # primeiro centroide: aleatorio uniforme
        centroids[0] = X[rng.integers(0, n_samples)]

        for k in range(1, self.n_clusters):
            # distancia minima quadrada ao centroide mais proximo ja escolhido
            dists = np.min(
                np.linalg.norm(X[:, None, :] - centroids[:k][None, :, :], axis=2) ** 2,
                axis=1,
            )
            probs = dists / dists.sum()
            centroids[k] = X[rng.choice(n_samples, p=probs)]
        return centroids

    def _assign(self, X, centroids):
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        return np.argmin(dists, axis=1)

    def _run_once(self, X, rng):
        centroids = self._init_centroids(X, rng)

        for _ in range(self.n_iters):
            labels = self._assign(X, centroids)
            new_centroids = np.array([
                X[labels == k].mean(axis=0) if np.any(labels == k) else centroids[k]
                for k in range(self.n_clusters)
            ])

            shift = np.linalg.norm(new_centroids - centroids)
            centroids = new_centroids
            if shift < self.tol:
                break

        labels = self._assign(X, centroids)
        inertia = np.sum(
            (X - centroids[labels]) ** 2
        )
        return centroids, labels, inertia

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X must be 2D.")
        if len(X) < self.n_clusters:
            raise ValueError("n_samples must be >= n_clusters.")

        rng = np.random.default_rng(self.random_state)
        best = None

        for _ in range(self.n_init):
            centroids, labels, inertia = self._run_once(X, rng)
            if best is None or inertia < best[2]:
                best = (centroids, labels, inertia)

        self.centroids_, self.labels_, self.inertia_ = best
        return self

    def predict(self, X):
        if self.centroids_ is None:
            raise ValueError("Call fit() before predict().")
        X = np.asarray(X, dtype=float)
        return self._assign(X, self.centroids_)

    def fit_predict(self, X):
        return self.fit(X).labels_

In [ ]:
X, y_true = datasets.make_blobs(
    n_samples=500, centers=4, n_features=2,
    cluster_std=0.8, random_state=42
)

km = KMeans(n_clusters=4, n_init=10, random_state=42)
km.fit(X)
print("Inercia:", km.inertia_)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap="tab10", s=15)
axes[0].set_title("Ground truth")

axes[1].scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="tab10", s=15)
axes[1].scatter(
    km.centroids_[:, 0], km.centroids_[:, 1],
    c="black", marker="X", s=150, label="centroides"
)
axes[1].set_title("K-Means clusters")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# metodo do cotovelo
inertias = []
ks = range(1, 9)
for k in ks:
    m = KMeans(n_clusters=k, n_init=5, random_state=42).fit(X)
    inertias.append(m.inertia_)

plt.figure()
plt.plot(list(ks), inertias, marker="o")
plt.xlabel("n_clusters")
plt.ylabel("inertia")
plt.title("Elbow method")
plt.show()